# Exercițiu Laborator 5 — Modele Markov ascunse (HMM)

## Enunțul problemei
Un profesor dă teste a căror dificultate reprezintă starea ascunsă a sistemului:
* **Stări ascunse:** $S = \{\text{Dificil (D)}, \text{Mediu (M)}, \text{Ușor (U)}\}$.
* **Distribuția inițială:** probabilitatea dificultății primului test este uniformă:
  $$\pi = \left[\frac{1}{3}, \frac{1}{3}, \frac{1}{3}\right].$$
* **Matricea de tranziție $A$** (rânduri și coloane în ordinea `D, M, U`):
  * din `D`: următorul test este `M` sau `U` cu probabilitate $0.5$ fiecare ($0$ pentru `D`);
  * din `M`: următorul test este `D` cu $0.5$, `M` cu $0.25$, `U` cu $0.25$;
  * din `U`: următorul test este `D` cu $0.5$, `M` cu $0.25$, `U` cu $0.25$.
  $$
  A = \begin{pmatrix}
  0 & 0.5 & 0.5 \\
  0.5 & 0.25 & 0.25 \\
  0.5 & 0.25 & 0.25
  \end{pmatrix}.
  $$
* **Observații (notele studentului):** $O = \{\text{FB}, \text{B}, \text{S}, \text{NS}\}$.
* **Matricea de emisie $B$** (rânduri: `D, M, U`; coloane: `FB, B, S, NS`):
  $$
  B = \begin{pmatrix}
  0.10 & 0.20 & 0.40 & 0.30 \\
  0.15 & 0.25 & 0.50 & 0.10 \\
  0.20 & 0.30 & 0.40 & 0.10
  \end{pmatrix}.
  $$

**Cerințe:**
* **a)** Utilizați librăria `hmmlearn` pentru a defini modelul Markov ascuns și desenați diagrama de stare.
* **b)** Pentru secvența observată `FB, FB, S, B, B, S, B, B, NS, B, B, S`, determinați cea mai probabilă secvență de dificultăți pentru testele corespunzătoare și probabilitatea acesteia.

## a) Definirea modelului HMM în `hmmlearn` și desenarea diagramei de stare

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Definirea parametrilor HMM
stari = ["Dificil (D)", "Mediu (M)", "Ușor (U)"]
stari_scurt = ["D", "M", "U"]
observatii_nume = ["FB", "B", "S", "NS"]

pi = np.array([1/3, 1/3, 1/3])

A = np.array([
    [0.0, 0.5,  0.5],
    [0.5, 0.25, 0.25],
    [0.5, 0.25, 0.25]
])

B = np.array([
    [0.10, 0.20, 0.40, 0.30],
    [0.15, 0.25, 0.50, 0.10],
    [0.20, 0.30, 0.40, 0.10]
])

# Configurarea modelului în hmmlearn
try:
    from hmmlearn import hmm
    HMMClass = getattr(hmm, "CategoricalHMM", getattr(hmm, "MultinomialHMM"))
    model = HMMClass(n_components=3, init_params="")
    model.startprob_ = pi
    model.transmat_ = A
    model.emissionprob_ = B
    hmmlearn_disponibil = True
    print("Modelul CategoricalHMM din hmmlearn a fost definit cu succes!")
except ImportError:
    hmmlearn_disponibil = False
    print("Notă: hmmlearn nu este instalat în acest mediu; se folosește implementarea exactă Viterbi/Forward echivalentă.")

# Desenarea diagramei de stare a lanțului Markov ascuns + emisii
G = nx.DiGraph()
for i, s in enumerate(stari_scurt):
    emisie_str = ", ".join(f"{obs}:{B[i, k]:.2f}" for k, obs in enumerate(observatii_nume))
    G.add_node(s, label=f"{s}\n(π={pi[i]:.2f})\n[{emisie_str}]")

for i, u in enumerate(stari_scurt):
    for j, v in enumerate(stari_scurt):
        if A[i, j] > 0:
            G.add_edge(u, v, weight=A[i, j])

pos = {"D": (0.5, 1.0), "M": (0.0, 0.0), "U": (1.0, 0.0)}
plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G, pos, node_size=3600, node_color=["#ffadad", "#ffd6a5", "#caffbf"], edgecolors="#333333", linewidths=1.5)
labels_noduri = nx.get_node_attributes(G, "label")
nx.draw_networkx_labels(G, pos, labels=labels_noduri, font_size=8.5, font_weight="bold")

# Desenăm arcele (curbate pentru a nu se suprapune cele bidirecționale)
for (u, v, d) in G.edges(data=True):
    rad = 0.0 if u == v else 0.18
    nx.draw_networkx_edges(
        G, pos, edgelist=[(u, v)],
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#495057",
        node_size=3600
    )

# Etichete manuale clare pentru probabilitățile de tranziție
edge_text = {
    ("D", "M"): (0.14, 0.56, "0.50"),
    ("M", "D"): (0.34, 0.44, "0.50"),
    ("D", "U"): (0.86, 0.56, "0.50"),
    ("U", "D"): (0.66, 0.44, "0.50"),
    ("M", "U"): (0.50, -0.09, "0.25"),
    ("U", "M"): (0.50, 0.09, "0.25"),
    ("M", "M"): (-0.15, 0.12, "0.25"),
    ("U", "U"): (1.15, 0.12, "0.25"),
}
for _, (x, y, txt) in edge_text.items():
    plt.text(x, y, txt, fontsize=10, fontweight="bold", color="#1d3557",
             bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))

plt.title("Diagrama de stare a Modelului Markov Ascuns (tranziții și emisii)", fontsize=12, fontweight="bold")
plt.axis("off")
plt.xlim(-0.3, 1.3)
plt.ylim(-0.25, 1.25)
plt.tight_layout()
plt.show()

## b) Decodarea Viterbi a secvenței observate

Secvența observată de $T = 12$ note este:
$$
O = (\text{FB}, \text{FB}, \text{S}, \text{B}, \text{B}, \text{S}, \text{B}, \text{B}, \text{NS}, \text{B}, \text{B}, \text{S}).
$$

Folosind algoritmul **Viterbi** (`model.decode(X, algorithm="viterbi")` în `hmmlearn`), obținem:
* **Cea mai probabilă secvență de dificultăți ($X^*$):**
  $$
  (\text{D}, \text{U}, \text{D}, \text{U}, \text{D}, \text{M}, \text{D}, \text{U}, \text{D}, \text{U}, \text{D}, \text{M})
  $$
  *(Notă: Datorită simetriei dintre primele două observații `FB, FB`, secvența `U, D, U, D, M, D, U, D, U, D, M` are exact aceeași probabilitate comună maximă ca `D, U, D, U, D, M, D, U, D, U, D, M`.)*
* **Probabilitatea comună maximă $P(X^*, O)$:**
  $$
  P(X^*, O) = 2.109375 \times 10^{-11} \quad (\log P(X^*, O) \approx -24.582044).
  $$
* **Probabilitatea condiționată a secvenței optime $P(X^* \mid O) = \frac{P(X^*, O)}{P(O)}$:**
  Deoarece probabilitatea totală a observațiilor (calculată cu algoritmul Forward) este $P(O) \approx 7.325943 \times 10^{-8}$, probabilitatea a posteriori a traseului Viterbi este $P(X^* \mid O) \approx 0.000288$ ($0.0288\%$).

In [ ]:
# Secvența observată
secventa_note = ["FB", "FB", "S", "B", "B", "S", "B", "B", "NS", "B", "B", "S"]
obs_idx = np.array([observatii_nume.index(n) for n in secventa_note]).reshape(-1, 1)

if hmmlearn_disponibil:
    log_prob_viterbi, stari_optime_idx = model.decode(obs_idx, algorithm="viterbi")
    log_prob_obs = model.score(obs_idx)
else:
    # Implementare exactă Viterbi și Forward
    T = len(obs_idx)
    N = len(stari_scurt)
    delta = np.zeros((T, N))
    psi = np.zeros((T, N), dtype=int)
    delta[0] = pi * B[:, obs_idx[0, 0]]
    for t in range(1, T):
        for j in range(N):
            scoruri = delta[t - 1] * A[:, j]
            psi[t, j] = np.argmax(scoruri)
            delta[t, j] = B[j, obs_idx[t, 0]] * np.max(scoruri)
    stari_optime_idx = np.zeros(T, dtype=int)
    stari_optime_idx[-1] = np.argmax(delta[-1])
    for t in range(T - 1, 0, -1):
        stari_optime_idx[t - 1] = psi[t, stari_optime_idx[t]]
    log_prob_viterbi = np.log(np.max(delta[-1]))

    alpha = np.zeros((T, N))
    alpha[0] = pi * B[:, obs_idx[0, 0]]
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = B[j, obs_idx[t, 0]] * np.sum(alpha[t - 1] * A[:, j])
    log_prob_obs = np.log(np.sum(alpha[-1]))

secventa_dificultati = [stari_scurt[i] for i in stari_optime_idx]
prob_comuna_max = np.exp(log_prob_viterbi)
prob_obs = np.exp(log_prob_obs)
prob_conditionata = prob_comuna_max / prob_obs

print("Secvența observată de note:      ", secventa_note)
print("Secvența optimă de dificultăți:  ", secventa_dificultati)
print(f"Log-probabilitate Viterbi:        {log_prob_viterbi:.6f}")
print(f"Probabilitate comună P(X*, O):    {prob_comuna_max:.10e}")
print(f"Probabilitate observații P(O):    {prob_obs:.10e}")
print(f"Probabilitate condiționată P(X*|O): {prob_conditionata:.6f}")